<!-- track-identity-card -->
# Corner-type profile table

| | |
|---|---|
| Pipeline step | `04_corner_type_profile.ipynb` |
| Manuscript section | 4.2 |
| Copied from | `notebooks/NB7A7_corner_type_profile_v2_2026-07-19.ipynb` |
| Source sha256 | `119c11b765118175258368201d46c7c7` |

**Reads**

- `data/features/driver_corner_matrix_<track>.parquet`

**Writes**

- `data/fingerprints/corner_type_profiles_<policy>/corner_type_pivot.parquet`
- `.../corner_type_profile_crosstrack.parquet`
- `.../corner_type_metrics.parquet`
- `.../nb7a7_info.json`

Produces the corner inventory, the type thresholds and the trail-braking shares reported in Section 4.2.

> Copied from the working notebook named above. Two changes were made to it: this identity card and the bootstrap cell that follows it were added, and the hard-coded data paths were replaced with the root that the bootstrap cell resolves. The analysis code is unchanged.


# FAZ 7A.7 — Viraj Tipi Bazli Surucu Profili
**Tezin en ozgun katkisi — hicbir akademik calisma ve ticari arac bunu yapmiyor.**

Mevcut: "Surucu X genel olarak Hiz Odakli"
Yeni: "Surucu X yavas virajlarda agresif ama hizli virajlarda temkinli"

NB07 `speed_class` (slow/medium/fast) + driver_corner_matrix → viraj tipine gore ayri fingerprint.

## Adim 1: Veri Yukleme

In [ ]:
# track-config-bootstrap
# Locates track/config.py, which resolves the data root at run time.
# Works from a flat layout (track/ beside the notebooks) and from the
# repository layout (src/track/ one level up). See track/config.py.
import sys as _sys, pathlib as _pl
_cands = []
for _p in [_pl.Path.cwd()] + list(_pl.Path.cwd().parents):
    _cands += [_p, _p / "src"]
for _c in _cands:
    if (_c / "track" / "config.py").is_file():
        _sys.path.insert(0, str(_c))
        break
else:
    raise RuntimeError(
        "track/config.py not found. Run this notebook from inside the repository, "
        "or add the directory holding track/ to sys.path."
    )
from track.config import PROJECT_ROOT as TRACK_ROOT
print("data root:", TRACK_ROOT)


In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  KOSU AYARI — SADECE BURAYI DEGISTIR             ║
# ╚══════════════════════════════════════════════════╝
POLICY = "P_EXC"   # "P_INC" = A-blogu DAHIL | "P_EXC" = A-blogu HARIC

# --- ORTAK KIMLIK POLITIKASI (NB11 v3 ile AYNI liste) ---
EXCLUDE_HARD = [
    "20240213_000000_ACAI",   # oyun-ici yapay surucu
    "20240308_ensemble",      # RL politika ciktisi
    "20240501_MPC",           # klasik kontrol baseline (veri kumesi belgesi)
]
ABLOCK = [
    "20240410_A_12_123",
    "20240410_A_21_231",
    "20240411_A_12_312",
]
assert POLICY in ("P_INC", "P_EXC"), "POLICY 'P_INC' ya da 'P_EXC' olmali"
EXCLUDE_IDS = EXCLUDE_HARD + ([] if POLICY == "P_INC" else ABLOCK)
POL_SUF = "_inc" if POLICY == "P_INC" else "_exc"

import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROJECT = TRACK_ROOT
FEATURES = PROJECT / "data" / "features"
FINGERPRINTS = PROJECT / "data" / "fingerprints"
FIG_DIR = PROJECT / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

TRACKS = ['barcelona', 'monza', 'red_bull_ring']

print(f"Kimlik politikasi: {POLICY}  ->  {len(EXCLUDE_IDS)} kimlik dislaniyor")

print("=" * 65)
print("  ADIM 1: VERI YUKLEME")
print("=" * 65)

# 1a: Viraj siniflarini oku + GLOBAL esiklerle yeniden siniflandir
corner_classes = {}
all_apex_speeds = []

# Önce tüm virajları topla
for track in TRACKS:
    candidates = list(FEATURES.glob(f"*{track}*corners*v3*"))
    if not candidates:
        continue
    corners = pd.read_parquet(candidates[0])
    for _, row in corners.iterrows():
        all_apex_speeds.append(row.get('apex_speed', 0))

# Global tercile esikleri
all_speeds = np.array(all_apex_speeds)
Q33, Q66 = np.percentile(all_speeds, 33), np.percentile(all_speeds, 66)
print(f"  Global esikler: slow < {Q33:.0f} km/h < medium < {Q66:.0f} km/h < fast")

# Şimdi her pistte global eşiklerle sınıfla
for track in TRACKS:
    candidates = list(FEATURES.glob(f"*{track}*corners*v3*"))
    if not candidates:
        continue
    corners = pd.read_parquet(candidates[0])
    
    mapping = {}
    for i, row in corners.iterrows():
        cid = int(row.get('corner_id', i + 1))
        spd = row.get('apex_speed', 0)
        if spd < Q33:
            mapping[cid] = 'slow'
        elif spd < Q66:
            mapping[cid] = 'medium'
        else:
            mapping[cid] = 'fast'
    
    corner_classes[track] = mapping
    dist = pd.Series(mapping.values()).value_counts().to_dict()
    print(f"  {track}: {len(corners)} viraj -> {dist}")

# 1b: Driver corner matrix'leri oku
matrices = {}
for track in TRACKS:
    f = FEATURES / f"driver_corner_matrix_{track}.parquet"
    if f.exists():
        _m = pd.read_parquet(f)
        _n0 = _m['driver_id'].nunique()
        _hit = sorted(set(_m['driver_id']) & set(EXCLUDE_IDS))
        _m = _m[~_m['driver_id'].isin(EXCLUDE_IDS)].copy()
        matrices[track] = _m
        print(f"\n  Matrix {track}: {_m.shape}  "
              f"(kimlik filtresi {_n0} -> {_m['driver_id'].nunique()}; cikarilan: {_hit or '-'})")

print(f"\n  Viraj sinifi olan pistler: {list(corner_classes.keys())}")
print(f"  Matrix olan pistler: {list(matrices.keys())}")

## Adim 2: Viraj Tipine Gore Metrik Ayirma
Her surucu icin slow/medium/fast virajlardaki performansi ayri hesapla.

In [ ]:
print("=" * 65)
print("  ADIM 2: VIRAJ TIPI BAZLI METRIKLER")
print("=" * 65)

# Matrix kolonlarindaki viraj bazli metrikler: T{n}_apex_speed, T{n}_braking_dist, ...
CORNER_METRICS = ['apex_speed', 'braking_dist', 'exit_speed', 'trail']

rows = []

for track in TRACKS:
    if track not in matrices or track not in corner_classes:
        continue
    
    mx = matrices[track]
    cc = corner_classes[track]
    
    # Viraj bazli kolonlari tara
    max_corner = max(cc.keys())
    
    for _, driver_row in mx.iterrows():
        driver_id = driver_row.get('driver_id', 'unknown')
        
        # Her viraj tipi icin metrikleri topla
        type_data = {'slow': [], 'medium': [], 'fast': []}
        
        for corner_num, speed_class in cc.items():
            corner_vals = {}
            for metric in CORNER_METRICS:
                col = f"T{corner_num}_{metric}"
                if col in mx.columns:
                    corner_vals[metric] = driver_row[col]
            
            if corner_vals and speed_class in type_data:
                type_data[speed_class].append(corner_vals)
        
        # Ortalama hesapla
        for stype, corners_data in type_data.items():
            if not corners_data:
                continue
            
            row = {
                'driver_id': driver_id,
                'track': track,
                'corner_type': stype,
                'n_corners': len(corners_data),
            }
            
            for metric in CORNER_METRICS:
                vals = [c[metric] for c in corners_data if metric in c and pd.notna(c[metric])]
                if vals:
                    if metric == 'trail':
                        row[f'{metric}_pct'] = np.mean([float(v) for v in vals])
                    else:
                        row[f'mean_{metric}'] = np.mean(vals)
                        row[f'std_{metric}'] = np.std(vals) if len(vals) > 1 else 0
            
            rows.append(row)

ct_df = pd.DataFrame(rows)
print(f"\n  Corner-type DataFrame: {ct_df.shape}")
print(f"  Sutunlar: {list(ct_df.columns)}")
print(f"\n  Dagilim:")
print(ct_df.groupby(['corner_type', 'track']).size().unstack(fill_value=0))
print(f"\n  Ornek (ilk 5):")
print(ct_df.head())

## Adim 3: Cross-Track Agregasyon
Ayni surucunun farkli pistlerdeki viraj tipi performanslarini birlestir.

In [ ]:
print("=" * 65)
print("  ADIM 3: CROSS-TRACK VIRAJ TIPI PROFILI")
print("=" * 65)

# Her surucu x viraj tipi icin cross-track ortalama
agg_rows = []

numeric_cols = [c for c in ct_df.columns if c.startswith('mean_') or c.startswith('std_') or c.endswith('_pct')]

for driver_id in ct_df['driver_id'].unique():
    d = ct_df[ct_df['driver_id'] == driver_id]
    
    for ctype in ['slow', 'medium', 'fast']:
        subset = d[d['corner_type'] == ctype]
        if subset.empty:
            continue
        
        row = {
            'driver_id': driver_id,
            'corner_type': ctype,
            'n_tracks': len(subset),
            'total_corners': subset['n_corners'].sum(),
        }
        
        for col in numeric_cols:
            vals = subset[col].dropna()
            if len(vals) > 0:
                row[col] = vals.mean()
        
        agg_rows.append(row)

profile_df = pd.DataFrame(agg_rows)
print(f"  Profil: {profile_df.shape}")

# Pivot: her surucu bir satir, kolonlar = corner_type x metric
pivot_rows = []
for driver_id in profile_df['driver_id'].unique():
    d = profile_df[profile_df['driver_id'] == driver_id]
    row = {'driver_id': driver_id}
    
    for ctype in ['slow', 'medium', 'fast']:
        subset = d[d['corner_type'] == ctype]
        if not subset.empty:
            s = subset.iloc[0]
            for col in numeric_cols:
                if col in s and pd.notna(s[col]):
                    row[f'{ctype}_{col}'] = s[col]
    
    pivot_rows.append(row)

pivot_df = pd.DataFrame(pivot_rows).set_index('driver_id')
print(f"  Pivot: {pivot_df.shape}")
print(f"  Kolonlar: {list(pivot_df.columns)}")

# Kac surucu tum 3 viraj tipinde veri var?
complete = pivot_df.dropna(thresh=len(pivot_df.columns) * 0.5)
print(f"\n  Yeterli veriye sahip surucu: {len(complete)}/{len(pivot_df)}")

## Adim 4: Viraj Tipi Fingerprint Karsilastirma
Ayni surucunun slow vs fast virajlardaki farki — "surus stili profili".

In [ ]:
print("=" * 65)
print("  ADIM 4: SLOW vs FAST FARKI")
print("=" * 65)

# Her surucu icin slow-fast delta hesapla
delta_rows = []

for driver_id in profile_df['driver_id'].unique():
    d = profile_df[profile_df['driver_id'] == driver_id]
    slow = d[d['corner_type'] == 'slow']
    fast = d[d['corner_type'] == 'fast']
    
    if slow.empty or fast.empty:
        continue
    
    row = {'driver_id': driver_id}
    s, f = slow.iloc[0], fast.iloc[0]
    
    if 'mean_apex_speed' in s and 'mean_apex_speed' in f:
        row['apex_speed_ratio'] = s['mean_apex_speed'] / f['mean_apex_speed'] if f['mean_apex_speed'] > 0 else np.nan
    if 'mean_braking_dist' in s and 'mean_braking_dist' in f:
        row['braking_delta'] = s['mean_braking_dist'] - f['mean_braking_dist']
    if 'trail_pct' in s and 'trail_pct' in f:
        row['trail_slow'] = s['trail_pct']
        row['trail_fast'] = f['trail_pct']
    
    delta_rows.append(row)

delta_df = pd.DataFrame(delta_rows)
print(f"  Delta analizi: {len(delta_df)} surucu")
if not delta_df.empty:
    print(f"\n  Apex speed ratio (slow/fast):")
    print(f"    min={delta_df['apex_speed_ratio'].min():.3f}")
    print(f"    max={delta_df['apex_speed_ratio'].max():.3f}")
    print(f"    mean={delta_df['apex_speed_ratio'].mean():.3f}")
    print(f"\n  Braking distance delta (slow - fast):")
    if 'braking_delta' in delta_df:
        print(f"    mean={delta_df['braking_delta'].mean():.1f}m")
    print(f"\n  Trail braking pct:")
    if 'trail_slow' in delta_df and 'trail_fast' in delta_df:
        print(f"    slow virajlar: {delta_df['trail_slow'].mean():.1%}")
        print(f"    fast virajlar: {delta_df['trail_fast'].mean():.1%}")

## Adim 5: Kaydet + Ozet

In [ ]:
# Kaydet
# v2: politika ekli cikti — iki politika birbirini EZMEZ, eski arsiv korunur
out_dir = FINGERPRINTS / ("corner_type_profiles" + POL_SUF)
out_dir.mkdir(parents=True, exist_ok=True)

ct_df.to_parquet(out_dir / "corner_type_metrics.parquet", index=False)
profile_df.to_parquet(out_dir / "corner_type_profile_crosstrack.parquet", index=False)
pivot_df.to_parquet(out_dir / "corner_type_pivot.parquet")
if not delta_df.empty:
    delta_df.to_parquet(out_dir / "corner_type_delta.parquet", index=False)

print("=" * 65)
print(f"  7A.7 VIRAJ TIPI PROFIL OZETI   [politika: {POLICY}]")
print("=" * 65)
print(f"\n  Virajlar: {sum(len(v) for v in corner_classes.values())} ({len(corner_classes)} pist)")
print(f"  Corner-type metrikler: {ct_df.shape}")
print(f"  Cross-track profil: {profile_df.shape}")
print(f"  Pivot matrix: {pivot_df.shape}")
print(f"  Delta analiz: {len(delta_df)} surucu")
print(f"\n  Kaydedilen dosyalar:")
for f in sorted(out_dir.glob("*.parquet")):
    print(f"    {f.name} ({f.stat().st_size/1024:.0f}KB)")
print(f"\n  SONRAKI: Radar chart gorsellestirilme + cluster analizi")


In [ ]:
# Tüm virajların apex_speed dağılımını gör
all_speeds = []
for track in TRACKS:
    candidates = list(FEATURES.glob(f"*{track}*corners*v3*"))
    if candidates:
        c = pd.read_parquet(candidates[0])
        for _, row in c.iterrows():
            all_speeds.append({'track': track, 'apex_speed': row.get('apex_speed', row.get('entry_speed', 0)),
                              'corner_id': row.get('corner_id', 0),
                              'current_class': row.get('speed_class', '?')})

speeds_df = pd.DataFrame(all_speeds)
print(f"Tüm virajlar: {len(speeds_df)}")
print(f"\nApex speed istatistikleri:")
print(speeds_df['apex_speed'].describe())
print(f"\nMevcut sınıflandırma:")
print(speeds_df['current_class'].value_counts())

# Yeni global eşikler öner
q33 = speeds_df['apex_speed'].quantile(0.33)
q66 = speeds_df['apex_speed'].quantile(0.66)
print(f"\nGlobal tercile eşikleri: slow < {q33:.0f} km/h < medium < {q66:.0f} km/h < fast")

# Yeni dağılım
speeds_df['new_class'] = pd.cut(speeds_df['apex_speed'], 
    bins=[-np.inf, q33, q66, np.inf], labels=['slow', 'medium', 'fast'])
print(f"\nYeni global dağılım:")
print(speeds_df.groupby(['new_class', 'track']).size().unstack(fill_value=0))

## Adim 6 — izlenebilirlik kaydi (v2)


In [ ]:
# v2: K1'in TUM tabani tek dosyaya — "her sayi bir dosyaya baglanir"
import json as _json
from datetime import datetime as _dt
_ct_dist = {t: pd.Series(list(m.values())).value_counts().to_dict() for t, m in corner_classes.items()}
_apex = speeds_df['apex_speed'].describe()
_info = {
    'notebook': 'NB7A7_corner_type_profile_v2',
    'run_timestamp': _dt.now().isoformat(timespec='seconds'),
    'identity_policy': POLICY,
    'ablock_excluded': (POLICY == 'P_EXC'),
    'exclude_hard': EXCLUDE_HARD,
    'exclude_ids': EXCLUDE_IDS,
    'n_drivers_per_track': {t: int(m['driver_id'].nunique()) for t, m in matrices.items()},
    'n_drivers_pivot': int(len(pivot_df)),
    'n_drivers_complete': int(len(complete)),
    'driver_ids_pivot': sorted(str(d) for d in pivot_df.index),
    'n_pivot_columns': int(len(pivot_df.columns)),
    'n_corners_total': int(sum(len(v) for v in corner_classes.values())),
    'n_corners_per_track': {t: int(len(v)) for t, v in corner_classes.items()},
    'corner_type_dist_per_track': _ct_dist,
    'corner_thresholds_kmh': {'q33': float(Q33), 'q66': float(Q66)},
    'apex_speed_stats': {k: float(_apex[k]) for k in ['mean', 'std', 'min', 'max']},
    'apex_speed_ratio': ({'mean': float(delta_df['apex_speed_ratio'].mean()),
                          'min': float(delta_df['apex_speed_ratio'].min()),
                          'max': float(delta_df['apex_speed_ratio'].max())}
                         if not delta_df.empty else None),
    'trail_pct_slow': float(delta_df['trail_slow'].mean()) if not delta_df.empty else None,
    'trail_pct_fast': float(delta_df['trail_fast'].mean()) if not delta_df.empty else None,
    'braking_delta_mean_m': float(delta_df['braking_delta'].mean()) if not delta_df.empty else None,
}
with open(out_dir / 'nb7a7_info.json', 'w', encoding='utf-8') as _f:
    _json.dump(_info, _f, indent=2, ensure_ascii=False)
print(f"Izlenebilirlik yazildi: {out_dir / 'nb7a7_info.json'}")
print(_json.dumps(_info, indent=1, ensure_ascii=False)[:900])
